# Gemini Website Summarizer

This notebook uses Gemini through an OpenAI-compatible client to fetch website text, build chat messages, summarize the site, and render the final answer as Markdown.

In [ ]:
# imports

import os
from dotenv import load_dotenv
from website_scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI


## Imports

- `os` reads environment variables.
- `load_dotenv` loads API keys from `.env`.
- `fetch_website_contents` reads clean text from a web page.
- `Markdown` and `display` render summaries in the notebook.
- `OpenAI` connects to Gemini's OpenAI-compatible endpoint.

In [ ]:
# Load environment variables in a file called .env

load_dotenv(override=True)

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
api_key = os.getenv('GOOGLE_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


## Load Gemini API Configuration

- Loads environment variables from `.env`.
- Sets Gemini's OpenAI-compatible base URL.
- Reads `GOOGLE_API_KEY`.
- Prints a basic warning if the key is missing or appears malformed.

In [ ]:
message = "Hello, GPT! This is my first ever message to you! Hi!"

messages = [{"role": "user", "content": message}]

messages

## Create a First Chat Message

- Defines a simple test message.
- Wraps it in the chat `messages` format expected by the API.
- Displays the message payload before sending it.

In [ ]:
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=api_key)

response = gemini.chat.completions.create(model="gemini-3.6-flash", messages=messages)
response.choices[0].message.content

## Send the First Gemini Request

- Creates the Gemini client.
- Sends the test message to `gemini-3.6-flash`.
- Returns the assistant's response content.

In [ ]:
websiteContents = fetch_website_contents("https://shashanks-portfolio.vercel.app")
print(websiteContents)

## Fetch Website Content

- Fetches readable text from the portfolio website.
- Prints the scraped content so you can inspect what will be sent to the model.

In [ ]:
system_prompt = """
You are a clear, concise website summarization assistant.
Summarize only the meaningful page content and ignore navigation menus, cookie banners, ads, repeated links, scripts, and boilerplate.
Return well-structured Markdown with:
- a brief overview
- key points
- news or announcements, if present
- a note when the page content is too thin or mostly navigation
Do not invent details that are not present in the website text.
""".strip()

user_prompt = """
Here is the extracted text from a website.
Summarize the site for a reader who has not opened it.
Focus on what the site is about, important offerings or projects, and any news or announcements.

Website content:
""".strip()

## Define Summary Prompts

- The system prompt sets the assistant's summarization behavior.
- The user prompt explains that website content will follow.
- The prompt asks for a grounded Markdown summary, key points, and any news or announcements.

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant that answers clearly and briefly"},
    {"role": "user", "content": "Who is the president of america?"}
]

response = gemini.chat.completions.create(model="gemini-3.6-flash", messages=messages)
response.choices[0].message.content

## Try a Custom Chat Prompt

- Builds a separate test conversation.
- Sends it to Gemini using the same client.
- Shows how changing messages changes the assistant's behavior.

In [ ]:
def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"{user_prompt}\n\n{website}"},
    ]

## Build Messages for Website Summaries

- `messages_for` accepts scraped website text.
- It combines the system prompt with the website-specific user prompt and scraped content.
- It returns the final messages list for the chat completion call.

In [ ]:
messages_for(websiteContents)

## Preview the Summary Messages

- Calls `messages_for` with the scraped website text.
- Lets you inspect the exact payload before sending it to Gemini.

In [ ]:
def summarize(url):
    website = fetch_website_contents(url)
    response = gemini.chat.completions.create(
        model = "gemini-3.6-flash",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

## Summarize Any URL

- `summarize` fetches a website from a URL.
- It sends the website content to Gemini.
- It returns the generated summary text.

In [ ]:
summarize("https://shashanks-portfolio.vercel.app")

## Run the Portfolio Summary

- Calls `summarize` on the portfolio website.
- Displays the raw returned summary as notebook output.

In [ ]:
def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

## Display Summaries as Markdown

- `display_summary` wraps the generated summary in `Markdown`.
- This makes headings, lists, and emphasis render cleanly in the notebook.

In [ ]:
display_summary("https://cnn.com")

## Try Another Website

- Runs the display helper on CNN.
- The rendered result shows how the summarizer behaves on a larger news site.